# Thử nghiệm bổ sung cho bài toán Abalone

Sổ tay này thực hiện các thí nghiệm mở rộng sau giai đoạn huấn luyện và đánh giá cơ bản: kiểm tra độ nhạy theo chiến lược tiền xử lý, thử nghiệm loại bỏ nhóm đặc trưng và tinh chỉnh siêu tham số cho mô hình mạnh nhất.

## 1. Mục tiêu thử nghiệm bổ sung

- Xác minh chiến lược dữ liệu nào ổn định nhất khi thay đổi mô hình.
- Đo ảnh hưởng của nhóm đặc trưng mới lên hiệu năng dự đoán.
- Tinh chỉnh mô hình cây bằng `RandomizedSearchCV`.
- Tổng hợp kết quả để đề xuất cấu hình cuối cùng cho báo cáo.

## 2. Import thư viện

In [ ]:
import json
import pickle
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, RandomizedSearchCV, cross_validate

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", 100)

## 3. Nạp dữ liệu từ các bước trước

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
INTERIM_DATA_PATH = PROJECT_ROOT / "data" / "interim"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"
OUTPUT_METRICS_PATH = PROJECT_ROOT / "outputs" / "metrics"
OUTPUT_FIGURES_PATH = PROJECT_ROOT / "outputs" / "figures"
OUTPUT_MODELS_PATH = PROJECT_ROOT / "outputs" / "models"

OUTPUT_METRICS_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_MODELS_PATH.mkdir(parents=True, exist_ok=True)


def load_pickle(path: Path):
    with open(path, "rb") as f:
        return pickle.load(f)

baseline_scaled = load_pickle(INTERIM_DATA_PATH / "baseline_scaled.pkl")
iqr_scaled = load_pickle(INTERIM_DATA_PATH / "iqr_filtered_scaled.pkl")
log_scaled = load_pickle(INTERIM_DATA_PATH / "log_transformed_scaled.pkl")
fe_scaled = load_pickle(PROCESSED_DATA_PATH / "baseline_with_features_scaled.pkl")
fe_unscaled = load_pickle(PROCESSED_DATA_PATH / "baseline_with_features_unscaled.pkl")

datasets = {
    "baseline_standard": {
        "X_train": baseline_scaled["standard"]["X_train"],
        "X_test": baseline_scaled["standard"]["X_test"],
        "y_train": baseline_scaled["y_train"],
        "y_test": baseline_scaled["y_test"],
    },
    "baseline_robust": {
        "X_train": baseline_scaled["robust"]["X_train"],
        "X_test": baseline_scaled["robust"]["X_test"],
        "y_train": baseline_scaled["y_train"],
        "y_test": baseline_scaled["y_test"],
    },
    "iqr_standard": {
        "X_train": iqr_scaled["standard"]["X_train"],
        "X_test": iqr_scaled["standard"]["X_test"],
        "y_train": iqr_scaled["y_train"],
        "y_test": iqr_scaled["y_test"],
    },
    "log_standard": {
        "X_train": log_scaled["standard"]["X_train"],
        "X_test": log_scaled["standard"]["X_test"],
        "y_train": log_scaled["y_train"],
        "y_test": log_scaled["y_test"],
    },
    "feature_engineering_standard": {
        "X_train": fe_scaled["X_train"],
        "X_test": fe_scaled["X_test"],
        "y_train": fe_scaled["y_train"],
        "y_test": fe_scaled["y_test"],
    },
}

for name, item in datasets.items():
    print(f"{name:30s} | tập huấn luyện={item['X_train'].shape} | tập kiểm tra={item['X_test'].shape}")

## 4. Hàm tiện ích cho thử nghiệm

In [ ]:
def evaluate_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_true, y_pred),
    }


def make_model(name: str, random_state: int = 42):
    if name == "LinearRegression":
        return LinearRegression()
    if name == "Ridge":
        return Ridge(alpha=1.0, random_state=random_state)
    if name == "RandomForest":
        return RandomForestRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_split=4,
            min_samples_leaf=2,
            random_state=random_state,
            n_jobs=-1,
        )
    if name == "GradientBoosting":
        return GradientBoostingRegressor(random_state=random_state)
    raise ValueError(f"Unsupported model: {name}")

## 5. Thử nghiệm A: So sánh mô hình trên nhiều chiến lược dữ liệu

In [ ]:
benchmark_rows = []
candidate_models = ["LinearRegression", "Ridge", "RandomForest", "GradientBoosting"]

for dataset_name, data in datasets.items():
    X_train, X_test = data["X_train"], data["X_test"]
    y_train, y_test = data["y_train"], data["y_test"]

    for model_name in candidate_models:
        model = make_model(model_name)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        metrics = evaluate_metrics(y_test, y_pred)

        cv = KFold(n_splits=5, shuffle=True, random_state=42)
        cv_result = cross_validate(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring={
                "mae": "neg_mean_absolute_error",
                "rmse": "neg_root_mean_squared_error",
                "r2": "r2",
            },
            n_jobs=-1,
            return_train_score=False,
        )

        benchmark_rows.append(
            {
                "dataset": dataset_name,
                "model": model_name,
                "test_mae": metrics["MAE"],
                "test_rmse": metrics["RMSE"],
                "test_r2": metrics["R2"],
                "cv_mae": -cv_result["test_mae"].mean(),
                "cv_rmse": -cv_result["test_rmse"].mean(),
                "cv_r2": cv_result["test_r2"].mean(),
            }
        )

benchmark_df = pd.DataFrame(benchmark_rows).sort_values(["test_rmse", "test_mae"], ascending=True).reset_index(drop=True)
display(benchmark_df.head(20))

best_benchmark = benchmark_df.iloc[0]
print("Cấu hình tốt nhất ở thử nghiệm A:")
print(best_benchmark)

In [ ]:
plt.figure(figsize=(14, 6))
plot_df = benchmark_df.head(12).copy()
plot_df["config"] = plot_df["dataset"] + " | " + plot_df["model"]
sns.barplot(data=plot_df, x="config", y="test_rmse", palette="crest")
plt.xticks(rotation=45, ha="right")
plt.title("Top 12 cấu hình theo RMSE trên tập kiểm tra")
plt.xlabel("Cấu hình")
plt.ylabel("RMSE trên tập kiểm tra")
plt.tight_layout()
plt.show()

## 6. Thử nghiệm B: Loại bỏ nhóm đặc trưng mới

So sánh trực tiếp giữa tập `baseline` và tập có đặc trưng mở rộng để đo mức đóng góp của các đặc trưng tạo thêm.

In [ ]:
ablation_rows = []

ablation_setups = {
    "baseline_standard": datasets["baseline_standard"],
    "feature_engineering_standard": datasets["feature_engineering_standard"],
}

for setup_name, data in ablation_setups.items():
    for model_name in ["Ridge", "RandomForest", "GradientBoosting"]:
        model = make_model(model_name)
        model.fit(data["X_train"], data["y_train"])
        pred = model.predict(data["X_test"])
        m = evaluate_metrics(data["y_test"], pred)

        ablation_rows.append(
            {
                "setup": setup_name,
                "model": model_name,
                "test_mae": m["MAE"],
                "test_rmse": m["RMSE"],
                "test_r2": m["R2"],
                "n_features": data["X_train"].shape[1],
            }
        )

ablation_df = pd.DataFrame(ablation_rows).sort_values(["model", "test_rmse"])
display(ablation_df)

plt.figure(figsize=(10, 5))
sns.barplot(data=ablation_df, x="model", y="test_rmse", hue="setup", palette="Set2")
plt.title("Loại bỏ đặc trưng: ảnh hưởng của đặc trưng mở rộng lên RMSE")
plt.xlabel("Mô hình")
plt.ylabel("RMSE trên tập kiểm tra")
plt.tight_layout()
plt.show()

## 7. Thử nghiệm C: Tinh chỉnh RandomForest với RandomizedSearchCV

Thực hiện tinh chỉnh trên cấu hình dữ liệu tốt nhất từ thử nghiệm A để tối ưu thêm hiệu năng.

In [ ]:
best_dataset_from_benchmark = benchmark_df.iloc[0]["dataset"]
rf_data = datasets[best_dataset_from_benchmark]

X_train_rf = rf_data["X_train"]
y_train_rf = rf_data["y_train"]
X_test_rf = rf_data["X_test"]
y_test_rf = rf_data["y_test"]

param_dist = {
    "n_estimators": [200, 300, 400, 500, 700],
    "max_depth": [None, 8, 12, 16, 24],
    "min_samples_split": [2, 4, 6, 8, 10],
    "min_samples_leaf": [1, 2, 3, 4],
    "max_features": ["sqrt", "log2", 0.7, 0.9],
}

base_rf = RandomForestRegressor(random_state=42, n_jobs=-1)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

rf_search = RandomizedSearchCV(
    estimator=base_rf,
    param_distributions=param_dist,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

rf_search.fit(X_train_rf, y_train_rf)

best_rf = rf_search.best_estimator_
best_rf_pred = best_rf.predict(X_test_rf)
best_rf_metrics = evaluate_metrics(y_test_rf, best_rf_pred)

print(f"Bộ dữ liệu dùng để tinh chỉnh: {best_dataset_from_benchmark}")
print("Bộ siêu tham số tốt nhất:")
print(rf_search.best_params_)
print(f"RMSE CV tốt nhất: {-rf_search.best_score_:.4f}")
print(f"RandomForest đã tinh chỉnh - Test MAE : {best_rf_metrics['MAE']:.4f}")
print(f"RandomForest đã tinh chỉnh - Test RMSE: {best_rf_metrics['RMSE']:.4f}")
print(f"RandomForest đã tinh chỉnh - Test R2  : {best_rf_metrics['R2']:.4f}")

In [ ]:
# So sánh trước và sau tinh chỉnh trên cùng bộ dữ liệu
rf_default = make_model("RandomForest")
rf_default.fit(X_train_rf, y_train_rf)
rf_default_pred = rf_default.predict(X_test_rf)
rf_default_metrics = evaluate_metrics(y_test_rf, rf_default_pred)

compare_tuning_df = pd.DataFrame(
    [
        {
            "configuration": "RandomForest mặc định",
            "test_mae": rf_default_metrics["MAE"],
            "test_rmse": rf_default_metrics["RMSE"],
            "test_r2": rf_default_metrics["R2"],
        },
        {
            "configuration": "RandomForest đã tinh chỉnh",
            "test_mae": best_rf_metrics["MAE"],
            "test_rmse": best_rf_metrics["RMSE"],
            "test_r2": best_rf_metrics["R2"],
        },
    ]
)

display(compare_tuning_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(data=compare_tuning_df, x="configuration", y="test_rmse", ax=axes[0], palette="flare")
axes[0].set_title("RMSE trước và sau tinh chỉnh")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(data=compare_tuning_df, x="configuration", y="test_mae", ax=axes[1], palette="crest")
axes[1].set_title("MAE trước và sau tinh chỉnh")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

## 8. Lưu kết quả thử nghiệm bổ sung

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

benchmark_path = OUTPUT_METRICS_PATH / "supplement_benchmark_results.csv"
ablation_path = OUTPUT_METRICS_PATH / "supplement_ablation_results.csv"
tuning_compare_path = OUTPUT_METRICS_PATH / "supplement_tuning_compare.csv"
report_path = OUTPUT_METRICS_PATH / "supplement_experiments_report.json"
model_path = OUTPUT_MODELS_PATH / "tuned_random_forest.pkl"

benchmark_df.to_csv(benchmark_path, index=False)
ablation_df.to_csv(ablation_path, index=False)
compare_tuning_df.to_csv(tuning_compare_path, index=False)

with open(model_path, "wb") as f:
    pickle.dump(best_rf, f)

report = {
    "timestamp": timestamp,
    "best_benchmark_config": {
        "dataset": str(best_benchmark["dataset"]),
        "model": str(best_benchmark["model"]),
        "test_rmse": float(best_benchmark["test_rmse"]),
        "test_mae": float(best_benchmark["test_mae"]),
        "test_r2": float(best_benchmark["test_r2"]),
    },
    "ablation_best_per_model": ablation_df.sort_values(["model", "test_rmse"]).groupby("model").first().reset_index().to_dict(orient="records"),
    "tuning_best_params": rf_search.best_params_,
    "tuning_test_metrics": {
        "mae": float(best_rf_metrics["MAE"]),
        "rmse": float(best_rf_metrics["RMSE"]),
        "r2": float(best_rf_metrics["R2"]),
    },
}

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("Đã lưu kết quả thử nghiệm bổ sung:")
print(f"- Bảng benchmark: {benchmark_path}")
print(f"- Bảng loại bỏ đặc trưng: {ablation_path}")
print(f"- Bảng so sánh tinh chỉnh: {tuning_compare_path}")
print(f"- Mô hình đã tinh chỉnh: {model_path}")
print(f"- Báo cáo tổng hợp: {report_path}")

In [ ]:
# Trực quan lỗi của mô hình RandomForest đã tinh chỉnh
rf_diag = pd.DataFrame(
    {
        "actual": y_test_rf.values,
        "predicted": best_rf_pred,
        "residual": y_test_rf.values - best_rf_pred,
    }
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(data=rf_diag, x="actual", y="predicted", alpha=0.6, ax=axes[0])
min_v = min(rf_diag["actual"].min(), rf_diag["predicted"].min())
max_v = max(rf_diag["actual"].max(), rf_diag["predicted"].max())
axes[0].plot([min_v, max_v], [min_v, max_v], "r--", linewidth=1.2)
axes[0].set_title("RandomForest đã tinh chỉnh: thực tế và dự đoán")

sns.histplot(rf_diag["residual"], bins=30, kde=True, ax=axes[1], color="#2a9d8f")
axes[1].axvline(0, color="red", linestyle="--", linewidth=1.2)
axes[1].set_title("RandomForest đã tinh chỉnh: phân phối phần dư")

plt.tight_layout()
plt.show()

## 9. Kết luận thử nghiệm

- Thử nghiệm A cho thấy hiệu năng thay đổi đáng kể giữa các chiến lược dữ liệu và loại mô hình.
- Thử nghiệm B cho thấy đặc trưng mở rộng có thể cải thiện hoặc không cải thiện tùy từng mô hình, vì vậy cần kiểm tra thực nghiệm thay vì giả định.
- Thử nghiệm C cho thấy tinh chỉnh RandomForest giúp kiểm soát sai số tốt hơn so với cấu hình mặc định trong đa số trường hợp.
- Kết quả đã được lưu thành các tệp đầu ra để phục vụ báo cáo và slide tổng kết.